# 第 5 章 · Tool Use（Executing Code / ADK）

比内建搜索更极端：模型**自己写 Python**，在服务端执行，再把 stdout 拿回来。

注意 API 形状——它**不在 `tools=[]` 里**，是单独的 `code_executor=` 字段。
下面把三样东西分开打印：`executable_code`（它写的代码）、`code_execution_result`（执行输出）、
最终 `text`。这三样的分离才是这一节的重点。

kernel `Python (agentic-adk)`。


In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, short_error, retry_call
from google.adk.agents import LlmAgent
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.runners import InMemoryRunner
from google.genai import types

USER_ID = "user1234"

# ★ 本章新增：code_executor 是独立字段，不是工具列表中的一项。
code_agent = LlmAgent(
    name="calculator_agent",
    model=adk_model(),                          # 【适配】gemini-2.0-flash 已下线
    code_executor=BuiltInCodeExecutor(),
    instruction="""You are a calculator agent.
   When given a mathematical expression, write and execute Python code to calculate the result.
   Return only the final numerical result as plain text, without markdown or code blocks.
   """,
    description="Executes Python code to perform calculations.",
)


def call_agent(query: str) -> None:
    runner = InMemoryRunner(code_agent)
    session_id = str(uuid.uuid4())
    # 【适配】create_session 是协程 → _sync 版
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id=USER_ID, session_id=session_id
    )
    print(f"\n--- Running Query: {query} ---")
    final = "No final text response captured."
    try:
        for event in retry_call(lambda: list(runner.run(
                user_id=USER_ID, session_id=session_id,
                new_message=types.Content(role="user", parts=[types.Part(text=query)]))), label="执行 "):
            for part in (event.content.parts or []) if event.content else []:
                if getattr(part, "executable_code", None):
                    print("  模型写的代码:\n```python\n" + (part.executable_code.code or "").strip() + "\n```")
                elif getattr(part, "code_execution_result", None):
                    r = part.code_execution_result
                    print(f"  执行结果: {getattr(r, 'outcome', '?')} -> {(r.output or '').strip()}")
                elif getattr(part, "text", None) and part.text.strip():
                    final = part.text.strip()
    except Exception as e:
        print(f"  ❌ {type(e).__name__}: {short_error(e)}")
        return
    print("==> Final Agent Response:", final)


call_agent("Calculate the value of (5 + 7) * 3")
call_agent("What is 10 factorial?")     # 10! 心算易错，能看出它是否真的执行了代码



--- Running Query: Calculate the value of (5 + 7) * 3 ---


  模型写的代码:
```python
print((5 + 7) * 3)
```
  执行结果: Outcome.OUTCOME_OK -> 36
==> Final Agent Response: 36

--- Running Query: What is 10 factorial? ---


  模型写的代码:
```python
import math
print(math.factorial(10))
```
  执行结果: Outcome.OUTCOME_OK -> 3628800
==> Final Agent Response: 3628800
